<a href="https://colab.research.google.com/github/irenetobby/irisclassifier_fastapi/blob/main/irisclassifier_fastapi_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import uvicorn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import threading
import time

app = FastAPI()

# Train and save a dummy model if model.pkl does not exist
try:
    with open("model.pkl", "rb") as f:
        model = pickle.load(f)
except FileNotFoundError:
    iris = load_iris()
    X, y = iris.data, iris.target
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)
    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)


class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float


@app.get("/")
def home():
    return {"message": "Iris Classifier API is running"}


@app.post("/predict")
def predict(data: IrisInput):

    features = [[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]]

    prediction = model.predict(features)[0]

    iris_classes = ["Setosa", "Versicolor", "Virginica"]
    result = iris_classes[prediction]

    return {"prediction": result}

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Run the Uvicorn server in a separate thread
uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.start()

# Give the server a moment to start up
time.sleep(1)
print("FastAPI server started in a separate thread on 127.0.0.1:8000")

INFO:     Started server process [294]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


FastAPI server started in a separate thread on 127.0.0.1:8000
